## 1. CARGA DE PARÁMETROS

In [0]:
import traceback

try:
    # Definimos valores por defecto para evitar errores de widget no definido
    dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
    dbutils.widgets.text("silver_schema", "sales_silver", "Silver Schema Name")
    dbutils.widgets.text("gold_schema", "sales_gold", "Gold Schema Name")
    dbutils.widgets.text("silver_table", "sales_cleaned", "Silver Table Name")
    dbutils.widgets.text("dim_vendor", "dim_vendor", "Vendor Dimension Table Name")
    dbutils.widgets.text("dim_store", "dim_store", "Store Dimension Table Name")
    dbutils.widgets.text("dim_product", "dim_product", "Product Dimension Table Name")

    env = {
        "catalog": dbutils.widgets.get("catalog"),
        "sch_silver": dbutils.widgets.get("silver_schema"),
        "sch_gold": dbutils.widgets.get("gold_schema"),
        "tb_silver": dbutils.widgets.get("silver_table"),
        "dim_vendor": dbutils.widgets.get("dim_vendor"),
        "dim_store": dbutils.widgets.get("dim_store"),
        "dim_product": dbutils.widgets.get("dim_product")
    }
    
    source_table = f"{env['catalog']}.{env['sch_silver']}.{env['tb_silver']}"
    print(f"Configuración cargada. Tabla origen: {source_table}")

except Exception as e:
    print("Error crítico al obtener los parámetros del notebook.")
    raise e

## 2. CREACIÓN DE LA TABLA DE HECHOS (DDL)

In [0]:
try:
    # DIM VENDOR (SCD1)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {env['catalog']}.{env['sch_gold']}.{env['dim_vendor']} (
      vendor_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      vendor_business_key INT,
      vendor_name STRING,
      saved_date TIMESTAMP
    ) USING DELTA COMMENT 'Vendor Dimension - SCD Type 1'
    """)
    
    # DIM PRODUCT (SCD1)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {env['catalog']}.{env['sch_gold']}.{env['dim_product']} (
      product_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      product_business_key INT,
      item_description STRING,
      pack INT,
      bottle_volume_ml DOUBLE,
      category_code INT,
      category_name STRING,
      saved_date TIMESTAMP
    ) USING DELTA COMMENT 'Product Dimension - SCD Type 1'
    """)

    # DIM STORE (SCD2)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {env['catalog']}.{env['sch_gold']}.{env['dim_store']} (
      store_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      store_business_key INT,
      store_name STRING,
      address STRING,
      city STRING,
      zip_code STRING,
      county_number INT,
      county STRING,
      valid_from TIMESTAMP,
      valid_to TIMESTAMP,
      is_current BOOLEAN,
      saved_date TIMESTAMP 
    ) USING DELTA COMMENT 'Store Dimension - SCD Type 2'
    """)
    
    print("Todas las tablas de dimensiones están listas en el esquema Gold.")

except Exception as e:
    print("Error creando las tablas de dimensiones.")
    raise e

## 3. CARGA DE DIMENSIONES (Lógica de MERGE)

In [0]:
# --- 3.1 CARGA DE VENDEDORES (SCD TIPO 1) ---
try:
    print(f"Poblando dimensión Vendor (SCD1)...")
    spark.sql(f"""
    MERGE INTO {env['catalog']}.{env['sch_gold']}.{env['dim_vendor']} AS target
    USING (
      SELECT * FROM (
        -- CORRECCIÓN AQUÍ: silver_saved_date AS saved_date
        SELECT vendor_number, vendor_name, silver_saved_date AS saved_date,
               ROW_NUMBER() OVER(PARTITION BY vendor_number ORDER BY date DESC) as rn
        FROM {source_table} WHERE vendor_number IS NOT NULL
      ) WHERE rn = 1
    ) AS source
    ON target.vendor_business_key = source.vendor_number
    WHEN MATCHED AND (target.vendor_name IS DISTINCT FROM source.vendor_name) THEN
      UPDATE SET target.vendor_name = source.vendor_name, target.saved_date = source.saved_date
    WHEN NOT MATCHED THEN
      INSERT (vendor_business_key, vendor_name, saved_date)
      VALUES (source.vendor_number, source.vendor_name, source.saved_date)
    """)
except Exception as e:
    raise e

# --- 3.2 CARGA DE PRODUCTOS (SCD TIPO 1) ---
try:
    print(f"Poblando dimensión Product (SCD1)...")
    spark.sql(f"""
    MERGE INTO {env['catalog']}.{env['sch_gold']}.{env['dim_product']} AS target
    USING (
      SELECT * FROM (
        -- CORRECCIÓN AQUÍ: silver_saved_date AS saved_date
        SELECT item_number, item_description, pack, bottle_volume_ml, category_code, category_name, silver_saved_date AS saved_date,
               ROW_NUMBER() OVER(PARTITION BY item_number ORDER BY date DESC) as rn
        FROM {source_table} WHERE item_number IS NOT NULL
      ) WHERE rn = 1
    ) AS source
    ON target.product_business_key = source.item_number
    WHEN MATCHED AND (
         target.item_description IS DISTINCT FROM source.item_description
      OR target.pack IS DISTINCT FROM source.pack
      OR target.bottle_volume_ml IS DISTINCT FROM source.bottle_volume_ml
      OR target.category_code IS DISTINCT FROM source.category_code
      OR target.category_name IS DISTINCT FROM source.category_name
    ) THEN
      UPDATE SET
        target.item_description = source.item_description, target.pack = source.pack,
        target.bottle_volume_ml = source.bottle_volume_ml, target.category_code = source.category_code,
        target.category_name = source.category_name, target.saved_date = source.saved_date
    WHEN NOT MATCHED THEN
      INSERT (product_business_key, item_description, pack, bottle_volume_ml, category_code, category_name, saved_date)
      VALUES (source.item_number, source.item_description, source.pack, source.bottle_volume_ml, source.category_code, source.category_name, source.saved_date)
    """)
except Exception as e:
    raise e

# --- 3.3 CARGA DE TIENDAS (SCD TIPO 2 - MANTENIENDO HISTORIAL) ---
try:
    print(f"Poblando dimensión Store (SCD2)...")
    
    spark.sql(f"""
    WITH source_data AS (
      SELECT * FROM (
        -- CORRECCIÓN AQUÍ: silver_saved_date AS saved_date
        SELECT store_number, store_name, address, city, zip_code, county_number, county, silver_saved_date AS saved_date,
               ROW_NUMBER() OVER(PARTITION BY store_number ORDER BY date DESC) as rn
        FROM {source_table} WHERE store_number IS NOT NULL
      ) WHERE rn = 1
    ),
    updates AS (
      SELECT s.*, t.store_key as merge_key
      FROM source_data s
      JOIN {env['catalog']}.{env['sch_gold']}.{env['dim_store']} t 
        ON s.store_number = t.store_business_key AND t.is_current = true
      WHERE s.store_name IS DISTINCT FROM t.store_name
         OR s.address IS DISTINCT FROM t.address
         OR s.city IS DISTINCT FROM t.city
         OR s.zip_code IS DISTINCT FROM t.zip_code
         OR s.county_number IS DISTINCT FROM t.county_number
         OR s.county IS DISTINCT FROM t.county
         
      UNION ALL
      
      SELECT s.*, NULL as merge_key
      FROM source_data s
    )
    
    MERGE INTO {env['catalog']}.{env['sch_gold']}.{env['dim_store']} AS target
    USING updates AS source
    ON target.store_key = source.merge_key
    
    WHEN MATCHED AND target.is_current = true THEN
      UPDATE SET 
        target.is_current = false, 
        target.valid_to = source.saved_date
        
    WHEN NOT MATCHED THEN
      INSERT (
        store_business_key, store_name, address, city, zip_code, county_number, county, 
        valid_from, valid_to, is_current, saved_date
      )
      VALUES (
        source.store_number, source.store_name, source.address, source.city, source.zip_code, source.county_number, source.county, 
        source.saved_date, CAST('9999-12-31' AS TIMESTAMP), true, source.saved_date
      )
    """)
    print("¡Dimensión Store SCD Tipo 2 poblada exitosamente!")
    
except Exception as e:
    print(f"Error cargando {env['dim_store']}")
    raise e